# 04 · Perfilamiento de Clusters e Inferencia
## Segmentación Inteligente de Pacientes — Comfama

| | |
|---|---|
| **Propósito** | Cargar el modelo registrado, asignar cluster a cada paciente, **interpretar** cada subgrupo y persistir las asignaciones. |
| **Entrada** | Modelo UC `<catalog>.<schema>.fenotipo_clustering` + tabla `fenotipo_features`. |
| **Salida** | Tabla Delta `<catalog>.<schema>.fenotipo_clusters` (consumible por la app / consulta médica). |

---
### 🎯 Qué produce este notebook
1. **Asignación** de cluster por paciente (inferencia con el modelo versionado).
2. **Perfil interpretable** de cada cluster (proporciones de hábitos/medicamentos y distribución de categóricas).
3. **Validación** contra la etiqueta `grupos`.
4. **Nombres de subgrupos** editables por el equipo clínico → trazabilidad clínica.

### Paso 1 · Parámetros y catálogo de variables

In [ ]:
dbutils.widgets.text("catalog", "main", "Catálogo")
dbutils.widgets.text("schema", "fenotipo", "Esquema")
dbutils.widgets.text("model_name", "main.fenotipo.fenotipo_clustering", "Nombre del modelo (UC)")
dbutils.widgets.text("model_alias", "", "Alias/versión (vacío = última)")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
MODEL_NAME = dbutils.widgets.get("model_name")
MODEL_ALIAS = dbutils.widgets.get("model_alias").strip()

FEATURES_TABLE = f"{CATALOG}.{SCHEMA}.fenotipo_features"
PRED_TABLE = f"{CATALOG}.{SCHEMA}.fenotipo_clusters"

import mlflow
import pandas as pd
import numpy as np
mlflow.set_registry_uri("databricks-uc")

NUMERIC_FEATURES = [
    "habitos_alim_cantidad",
    "hab_dejar_endulzar", "hab_dejar_reposteria", "hab_incorporar_vegetales",
    "hab_ajustar_carbohidratos", "hab_aumentar_proteina", "hab_otros",
    "med_hipoglicemiantes", "med_hipolipemiantes", "med_ninguno",
]
CATEGORICAL_FEATURES = [
    "actividad_tipo", "actividad_duracion", "actividad_frecuencia",
    "estres_altas_cargas", "estres_tecnica_manejo",
]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

### Paso 2 · Cargar el modelo e inferir
Cargamos el modelo desde Unity Catalog (última versión o un alias específico) y asignamos cluster a cada paciente.

In [ ]:
model_uri = f"models:/{MODEL_NAME}@{MODEL_ALIAS}" if MODEL_ALIAS else f"models:/{MODEL_NAME}/latest"
print("Cargando:", model_uri)
model = mlflow.sklearn.load_model(model_uri)

pdf = spark.table(FEATURES_TABLE).toPandas()
X = pdf[ALL_FEATURES].copy()
for c in NUMERIC_FEATURES:
    X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)

pdf["cluster"] = model.predict(X)
print("Distribución de clusters:")
print(pdf["cluster"].value_counts().sort_index().to_string())

### Paso 3 · Perfil de variables binarias/numéricas
Para las binarias, el **promedio por cluster = proporción** de pacientes con ese hábito/medicamento. Es la base para **nombrar** cada subgrupo.

In [ ]:
perfil_num = pdf.groupby("cluster")[NUMERIC_FEATURES].apply(
    lambda g: g.apply(pd.to_numeric, errors="coerce").mean()
).round(3)
perfil_num.insert(0, "n_pacientes", pdf.groupby("cluster").size())
display(perfil_num.reset_index())

### Paso 4 · Perfil de variables categóricas (% por cluster)

In [ ]:
for c in CATEGORICAL_FEATURES:
    print(f"\n===== {c} (% por cluster) =====")
    tab = pd.crosstab(pdf["cluster"], pdf[c], normalize="index").mul(100).round(1)
    display(tab)

### Paso 5 · Validación de referencia
Cruce con la etiqueta `grupos`. Recordar que la cohorte es ~94% Cardiometabólico, por lo que este cruce funciona como **control**, no como objetivo.

In [ ]:
display(pd.crosstab(pdf["cluster"], pdf["grupos"]).reset_index())

### Paso 6 · Nombrar los subgrupos (trazabilidad clínica)
Tras leer los perfiles anteriores, el equipo clínico completa este diccionario con nombres accionables. Queda documentado en el propio notebook.

In [ ]:
ETIQUETAS_CLUSTER = {
    # 0: "Cardiometabólico – hábitos saludables activos",
    # 1: "Cardiometabólico – sin cambios de hábito / sin medicación",
    # 2: "Cardiometabólico – en manejo farmacológico",
    # 3: "Cardiometabólico – estrés no gestionado",
}
pdf["subgrupo"] = pdf["cluster"].map(ETIQUETAS_CLUSTER).fillna(
    pdf["cluster"].apply(lambda k: f"Subgrupo {k}")
)

### Paso 7 · Persistencia de asignaciones (capa *gold*)

In [ ]:
salida = pdf[["id", "fecha_atencion", "grupos", "cluster", "subgrupo"] + ALL_FEATURES].copy()
for col in salida.columns:
    if salida[col].dtype == "object":
        salida[col] = salida[col].astype(str)

sdf = spark.createDataFrame(salida)
(sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(PRED_TABLE))
print(f"✅ Asignaciones guardadas: {PRED_TABLE}  ({sdf.count():,} filas)")
display(spark.table(PRED_TABLE).limit(20))

### Paso 8 · (Opcional) Inferencia para un paciente nuevo
Demuestra cómo puntuar un registro individual con el mismo modelo — patrón reutilizable para integrarlo con la app de Streamlit.

In [ ]:
ejemplo = pd.DataFrame([{
    "habitos_alim_cantidad": 2,
    "hab_dejar_endulzar": 1, "hab_dejar_reposteria": 0, "hab_incorporar_vegetales": 1,
    "hab_ajustar_carbohidratos": 0, "hab_aumentar_proteina": 0, "hab_otros": 0,
    "med_hipoglicemiantes": 0, "med_hipolipemiantes": 1, "med_ninguno": 0,
    "actividad_tipo": "Cardiovascular (caminar, trotar, spinning)",
    "actividad_duracion": "Más de 30 minutos",
    "actividad_frecuencia": "3 veces",
    "estres_altas_cargas": "Sí",
    "estres_tecnica_manejo": "No",
}])
print("Cluster asignado al paciente de ejemplo:", int(model.predict(ejemplo)[0]))

---
**Resumen del notebook 04:** clusters asignados, interpretados y persistidos en `fenotipo_clusters`, listos para consumo en la app / consulta médica.